# 03 — Preprocessing & Feature/Graph Engineering for GNN

Notebook **3 of 4**. Builds the richer node-feature matrix *and* the transaction-transaction graph that
[04_gcn_model.ipynb](04_gcn_model.ipynb) trains a GCN on. This is a pure pandas/numpy notebook — no `torch`
dependency here, so it runs in any environment; notebook 4 converts the saved arrays to tensors.

Design choices (see [../README.md](../README.md) for the original rationale this generalizes):

- **Nodes** = transactions, from **train + test combined** (transductive graph — the GCN can use test-node
  connectivity/features at train time, it just never sees test *labels*). This is standard practice for
  graph-based Kaggle solutions on this dataset.
- **Edges** = shared `card1`-`card6`, `addr1`-`addr2`, `P_emaildomain`/`R_emaildomain`, `DeviceType`/`DeviceInfo`
  — motivated by [01_eda.ipynb](01_eda.ipynb) §13, which found larger shared-attribute groups have different
  fraud rates than singleton transactions.
- **Train/valid/test split**: reuses `fraud_utils.time_based_split` on the *labeled* rows only — identical
  split to [02_baseline_model.ipynb](02_baseline_model.ipynb), so the comparison in notebook 4 is apples-to-apples.
  Feature scaling statistics are fit on the `train` split only (not `valid`/`test`) to avoid leakage.

Outputs (written to `../ieee-fraud-detection/processed/`): `features.npy`, `labels.npy`, `split.npy`,
`edge_index.npy`, `edge_type.npy`, `feature_names.json`, `node_ids.parquet`.


In [1]:
import sys, pathlib, json
sys.path.append(str(pathlib.Path.cwd()))

import numpy as np
import pandas as pd

from fraud_utils import (DATA_DIR, PROCESSED_DIR, RANDOM_SEED, set_seed,
                          load_raw_data, time_based_split)

set_seed()
pd.options.display.max_columns = 100


## 1. Load & Merge Data

In [2]:
train, test = load_raw_data()


Loading train_transaction.csv ...
Loading train_identity.csv ...
Loading test_transaction.csv ...
Loading test_identity.csv ...
Reducing memory usage...
  Memory: 1984.2 MB -> 1073.5 MB (46% reduction)
  Memory: 1700.3 MB -> 922.3 MB (46% reduction)
train: (590540, 434), test: (506691, 433), fraud rate: 0.0350


## 2. Time-Based Split (on labeled rows), Then Combine With Test

We compute the train/valid split on `train` **first** (identical to notebook 2), then concatenate with
`test` into one combined node set. Every node gets a `split` label of `"train"`, `"valid"`, or `"test"`,
which notebook 4 turns directly into PyG boolean masks.


In [3]:
train_mask, valid_mask = time_based_split(train)

train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

train["split"] = np.where(train_mask, "train", "valid")
test["split"] = "test"
train["is_train"] = 1
test["is_train"] = 0
test["isFraud"] = -1  # unlabeled placeholder; never used as a supervision signal

df = pd.concat([train, test], axis=0, ignore_index=True)
df["node_id"] = np.arange(len(df))
print(df["split"].value_counts())
print(f"combined node count: {len(df):,}")


/var/tmp/ipykernel_5036/3432963622.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["split"] = np.where(train_mask, "train", "valid")
/var/tmp/ipykernel_5036/3432963622.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["split"] = "test"
/var/tmp/ipykernel_5036/3432963622.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented 

split
test     506691
train    472432
valid    118108
Name: count, dtype: int64
combined node count: 1,097,231


/var/tmp/ipykernel_5036/3432963622.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["node_id"] = np.arange(len(df))


## 3. GNN-Oriented Feature Engineering

Goes beyond the notebook-2 baseline: full variance-ranked V-column selection, per-card amount z-scores,
email-domain grouping, cyclical time encoding, and explicit shared-attribute group-size features (motivated
directly by the EDA finding that group size correlates with fraud rate — giving the GCN's input layer that
signal too, on top of what message passing discovers structurally).


In [4]:
EDGE_TYPE_COLS = ["card1", "card2", "card3", "card4", "card5", "card6",
                   "addr1", "addr2", "P_emaildomain", "R_emaildomain",
                   "DeviceType", "DeviceInfo"]


def engineer_gnn_features(df, train_split_mask):
    df = df.copy()

    # --- Transaction amount ---
    df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])
    decimal = df["TransactionAmt"] - df["TransactionAmt"].astype(int)
    df["TransactionAmt_decimal"] = decimal
    df["TransactionAmt_is_round"] = (decimal == 0).astype(int)

    # --- Time features (cyclical hour encoding) ---
    df["Transaction_hour"] = (df["TransactionDT"] / 3600) % 24
    df["Transaction_day"] = (df["TransactionDT"] / (3600 * 24)).astype(int)
    df["Transaction_dow"] = df["Transaction_day"] % 7
    df["hour_sin"] = np.sin(2 * np.pi * df["Transaction_hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["Transaction_hour"] / 24)

    # --- Shared-attribute group-size features (explicit signal for the same structure edges encode) ---
    for col in ["card1", "card2", "card3", "card4", "card5", "card6", "addr1", "addr2"]:
        vc = df[col].value_counts(dropna=False)
        df[f"{col}_count"] = df[col].map(vc)

    # --- Card1 x amount interaction, fit on train split only ---
    card1_amt = (df.loc[train_split_mask].groupby("card1")["TransactionAmt"]
                 .agg(["mean", "std"]).reset_index())
    card1_amt.columns = ["card1", "card1_amt_mean", "card1_amt_std"]
    df = df.merge(card1_amt, on="card1", how="left")
    df["card1_amt_zscore"] = (df["TransactionAmt"] - df["card1_amt_mean"]) / (df["card1_amt_std"] + 1e-8)

    # --- Email domain features ---
    for col in ["P_emaildomain", "R_emaildomain"]:
        df[col] = df[col].fillna("unknown")
        vc = df[col].value_counts()
        df[f"{col}_count"] = df[col].map(vc)
        df[f"{col}_group"] = df[col].apply(lambda x: x.split(".")[-1] if isinstance(x, str) else "unknown")
    df["email_match"] = (df["P_emaildomain"] == df["R_emaildomain"]).astype(int)

    # --- V-columns: keep top-50 by variance among <50% missing (fit on train split) ---
    v_cols = [c for c in df.columns if c.startswith("V") and c[1:].isdigit()]
    v_null_rate = df.loc[train_split_mask, v_cols].isnull().mean()
    v_candidates = v_null_rate[v_null_rate < 0.5].index
    v_variance = df.loc[train_split_mask, v_candidates].var()
    v_keep = v_variance.nlargest(50).index.tolist()
    print(f"Keeping {len(v_keep)} of {len(v_cols)} V-columns")

    # --- C / D columns (kept as-is) ---
    c_cols = [c for c in df.columns if c.startswith("C") and c[1:].isdigit()]
    d_cols = [c for c in df.columns if c.startswith("D") and c[1:].isdigit()]

    # --- M columns: T/F -> 1/0, missing -> -1 ---
    m_cols = [c for c in df.columns if c.startswith("M") and c[1:].isdigit()]
    for col in m_cols:
        df[col] = df[col].map({"T": 1, "F": 0}).fillna(-1)

    # --- Categorical label encoding (fit on combined data; no label information involved) ---
    cat_cols = ["ProductCD", "card4", "card6", "P_emaildomain_group", "R_emaildomain_group", "DeviceType"]
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype("category").cat.codes

    feature_cols = (
        ["TransactionAmt", "TransactionAmt_log", "TransactionAmt_decimal", "TransactionAmt_is_round",
         "Transaction_hour", "Transaction_day", "Transaction_dow", "hour_sin", "hour_cos",
         "card1_count", "card2_count", "card3_count", "card4_count", "card5_count", "card6_count",
         "card1_amt_mean", "card1_amt_std", "card1_amt_zscore",
         "P_emaildomain_count", "R_emaildomain_count", "email_match",
         "addr1_count", "addr2_count",
         "ProductCD", "card4", "card6", "P_emaildomain_group", "R_emaildomain_group", "DeviceType"]
        + v_keep + c_cols + d_cols + m_cols
    )
    feature_cols = [c for c in feature_cols if c in df.columns]
    print(f"Total engineered features: {len(feature_cols)}")

    return df, feature_cols


train_split_mask = (df["split"] == "train").values
df, feature_cols = engineer_gnn_features(df, train_split_mask)


Keeping 50 of 339 V-columns
Total engineered features: 117


In [5]:
# --- Fill missing, then standardize (fit stats on train split only -> no leakage into valid/test) ---
feature_df = df[feature_cols].copy().fillna(-999).astype(np.float64)

train_stats = feature_df.loc[train_split_mask]
means, stds = train_stats.mean(), train_stats.std() + 1e-8
feature_df = (feature_df - means) / stds

labels = df["isFraud"].values.astype(np.int64)
split_arr = df["split"].values

print(feature_df.shape, labels.shape, split_arr.shape)
assert not np.isnan(feature_df.values).any(), "NaNs remain in feature matrix"
assert not np.isinf(feature_df.values).any(), "Infs remain in feature matrix"


(1097231, 117) (1097231,) (1097231,)


## 4. Graph Construction — Shared-Attribute Edges

For each edge-type column, group transactions by value and connect same-group nodes. Groups larger than
`STAR_THRESHOLD` use a **star topology** (all nodes connect to one hub) instead of full pairwise connections,
to avoid O(n²) edge blow-up on huge card/address groups; groups larger than `MAX_GROUP_SIZE` are subsampled
first. This matches [../README.md](../README.md)'s architecture description and `../gnn_fraud_detection.py`.


In [6]:
MAX_GROUP_SIZE = 50    # cap group size before connecting (avoids O(n^2) on huge groups)
STAR_THRESHOLD = 20    # above this, use a star (hub) topology instead of full pairwise


def build_graph_edges(df, edge_type_cols, max_group_size=MAX_GROUP_SIZE, star_threshold=STAR_THRESHOLD):
    rng = np.random.default_rng(RANDOM_SEED)
    all_src, all_dst, all_types = [], [], []

    for etype_idx, col in enumerate(edge_type_cols):
        if col not in df.columns:
            continue
        edge_count = 0
        for val, indices in df.groupby(col).groups.items():
            if pd.isna(val):
                continue
            indices = indices.to_numpy()
            if len(indices) < 2:
                continue
            if len(indices) > max_group_size:
                indices = rng.choice(indices, max_group_size, replace=False)

            if len(indices) > star_threshold:
                hub = indices[0]
                others = indices[1:]
                all_src.extend([hub] * len(others)); all_dst.extend(others.tolist())
                all_src.extend(others.tolist()); all_dst.extend([hub] * len(others))
                all_types.extend([etype_idx] * (2 * len(others)))
                edge_count += 2 * len(others)
            else:
                for i in range(len(indices)):
                    for j in range(i + 1, len(indices)):
                        all_src.extend([indices[i], indices[j]])
                        all_dst.extend([indices[j], indices[i]])
                        all_types.extend([etype_idx, etype_idx])
                        edge_count += 2
        print(f"  {col:16s} -> {edge_count:>10,d} edges")

    edge_index = np.array([all_src, all_dst], dtype=np.int64)
    edge_type = np.array(all_types, dtype=np.int64)
    return edge_index, edge_type


edge_index, edge_type = build_graph_edges(df, EDGE_TYPE_COLS)
print(f"\nTotal edges: {edge_index.shape[1]:,}   avg degree: {edge_index.shape[1] / len(df):.1f}")


  card1            ->    899,694 edges
  card2            ->     49,098 edges
  card3            ->      8,842 edges
  card4            ->        490 edges
  card5            ->      9,360 edges
  card6            ->        592 edges
  addr1            ->     16,162 edges
  addr2            ->      3,906 edges
  P_emaildomain    ->      5,882 edges
  R_emaildomain    ->      6,168 edges
  DeviceType       ->        294 edges
  DeviceInfo       ->    160,910 edges

Total edges: 1,161,398   avg degree: 1.1


## 5. Save Preprocessed Artifacts

In [7]:
np.save(PROCESSED_DIR / "features.npy", feature_df.values.astype(np.float32))
np.save(PROCESSED_DIR / "labels.npy", labels)
np.save(PROCESSED_DIR / "split.npy", split_arr)
np.save(PROCESSED_DIR / "edge_index.npy", edge_index)
np.save(PROCESSED_DIR / "edge_type.npy", edge_type)

with open(PROCESSED_DIR / "feature_names.json", "w") as f:
    json.dump(feature_cols, f, indent=2)

df[["node_id", "TransactionID", "split"]].to_parquet(PROCESSED_DIR / "node_ids.parquet", index=False)

print(f"Artifacts written to {PROCESSED_DIR}")
for p in sorted(PROCESSED_DIR.iterdir()):
    print(f"  {p.name:24s} {p.stat().st_size / 1024**2:8.1f} MB")


Artifacts written to /home/jupyter/GNN-Fraud-Detection/ieee-cis-dataset/processed
  edge_index.npy               17.7 MB
  edge_type.npy                 8.9 MB
  feature_names.json            0.0 MB
  features.npy                489.7 MB
  labels.npy                    8.4 MB
  node_ids.parquet              9.4 MB
  split.npy                     7.9 MB


## 6. Sanity Checks

Reload from disk and verify shapes align and the fraud rates per split match notebook 2's split exactly
(they should, since both use `fraud_utils.time_based_split` with the same default fraction/seed).

In [8]:
feat_check = np.load(PROCESSED_DIR / "features.npy")
labels_check = np.load(PROCESSED_DIR / "labels.npy")
split_check = np.load(PROCESSED_DIR / "split.npy", allow_pickle=True)
ei_check = np.load(PROCESSED_DIR / "edge_index.npy")

assert feat_check.shape[0] == labels_check.shape[0] == split_check.shape[0] == len(df)
assert ei_check.max() < len(df)

for s in ["train", "valid", "test"]:
    m = split_check == s
    if s == "test":
        print(f"{s:6s}: {m.sum():>8,d} nodes (unlabeled)")
    else:
        print(f"{s:6s}: {m.sum():>8,d} nodes, fraud rate {labels_check[m].mean():.4f}")

print(f"\nnode_features: {feat_check.shape}, edges: {ei_check.shape[1]:,}, "
      f"features: {len(feature_cols)}")


train :  472,432 nodes, fraud rate 0.0351
valid :  118,108 nodes, fraud rate 0.0344
test  :  506,691 nodes (unlabeled)

node_features: (1097231, 117), edges: 1,161,398, features: 117


## Summary & Next Step

We now have, on disk in `../ieee-fraud-detection/processed/`:
- A `[n_nodes, n_features]` standardized feature matrix (`features.npy`)
- Labels with `-1` for unlabeled test nodes (`labels.npy`)
- A `train`/`valid`/`test` split array matching notebook 2's split exactly (`split.npy`)
- A transaction-transaction graph from shared card/address/email/device attributes (`edge_index.npy`,
  `edge_type.npy`)

**Next**: [04_gcn_model.ipynb](04_gcn_model.ipynb) loads these artifacts, trains a GCN, and compares it against
the notebook-2 baseline on the shared validation split.
